# Stage 12 — Inventory Policy: ROL / ROQ / Buffer
**Dashboard pages:** Order Plan (Tabs: Order Plan · Sanity Review · UIO-Based Plan) · Purchase Recommendation

**ROL** = 3-month forecast + safety stock
**SS**  = ML quantile regression (≥6 active months) or classical z×σ
**ROQ** = EOQ (critical/managed) | cover-period (watch) | min viable (rationalise)
**EOQ** = sqrt(2 × D_annual × 5000 LKR / (unit_value × 0.20))

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
pol = load("inventory_policy.parquet")
uio_plan = load("uio_based_demand.parquet") if (INTERIM/"uio_based_demand.parquet").exists() else pd.DataFrame()

print(f"Policy table  : {len(pol):,} SKUs")
print(f"UIO plan      : {len(uio_plan):,} SKUs")
print()
print("Policy tier distribution:")
print(pol["policy_tier"].value_counts().to_string())
print()
print("Order urgency distribution:")
print(pol["order_urgency"].value_counts().to_string())


## Order Plan — ROL / ROQ / Safety Stock

In [ ]:
tier_col = "policy_tier"; urg_col = "order_urgency"
fig,axes = plt.subplots(1,3,figsize=(15,4))
for ax,(col,lbl) in zip(axes,[("rol","Reorder Level (ROL)"),
                                ("safety_stock","Safety Stock"),("roq","Reorder Qty (ROQ)")]):
    d = pol[col].clip(upper=pol[col].quantile(0.95))
    d[d>0].hist(bins=60,ax=ax,color=PALETTE[0],edgecolor="white",alpha=0.8)
    ax.set_title(lbl); ax.set_ylabel("SKUs")
plt.tight_layout(); plt.show()

# Avg metrics by tier
print("Average policy metrics by tier:")
print(pol.groupby(tier_col)[["rol","roq","safety_stock","net_requirement","forecast_lt"]].mean().round(2).to_string())


## Order Urgency (filter controls on dashboard)

In [ ]:
urg_order = ["immediate","soon","planned","none"]
urg = pol["order_urgency"].value_counts().reindex([x for x in urg_order if x in pol["order_urgency"].values],fill_value=0)
urg_colors = ["#EF4444","#F97316","#2CC56F","#94A3B8"]
fig,ax = plt.subplots(figsize=(8,4))
urg.plot(kind="bar",ax=ax,color=urg_colors[:len(urg)],edgecolor="white")
ax.set_title("Order Urgency — Current Planning Cycle
"
             "(Immediate: stock<ROL and coverage<1m; Soon: 1-3m; Planned: >3m)")
ax.set_ylabel("SKUs"); ax.tick_params(axis="x",rotation=0)
for bar,val in zip(ax.patches,urg.values):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+2,str(val),ha="center",fontsize=10)
plt.tight_layout(); plt.show()

# ROL vs stock on hand
fig,ax = plt.subplots(figsize=(9,7))
for t,tc in TIER_COLORS.items():
    m = pol["policy_tier"]==t
    if m.sum():
        lim = max(pol["rol"].quantile(0.95),pol["stock_on_hand"].quantile(0.95))
        ax.scatter(pol.loc[m,"rol"].clip(upper=lim),pol.loc[m,"stock_on_hand"].clip(upper=lim),
                   alpha=0.35,s=18,color=tc,label=t,zorder=3)
ax.plot([0,lim],[0,lim],"--",color="gray",alpha=0.4,lw=1.5,label="Stock=ROL (reorder signal)")
ax.set_title("ROL vs Current Stock — SKUs below diagonal have stock<ROL
(reorder should be triggered)")
ax.set_xlabel("Reorder Level"); ax.set_ylabel("Stock on Hand"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

below_rol = (pol["stock_on_hand"]<pol["rol"]).sum()
print(f"SKUs below ROL (reorder triggered): {below_rol:,} ({below_rol/len(pol)*100:.1f}%)")


## Sanity Review Tab — Flagged SKUs

In [ ]:
if "sanity_flag" in pol.columns:
    flagged = pol[pol["sanity_flag"]==True] if pol["sanity_flag"].dtype==bool               else pol[pol["sanity_flag"]==1]
    print(f"Sanity-flagged SKUs: {len(flagged):,} of {len(pol):,}")
    if len(flagged):
        note_col = "sanity_note" if "sanity_note" in flagged.columns else None
        if note_col:
            print("
Flag reason distribution:")
            print(flagged[note_col].value_counts().to_string())
        display_cols = ["material_9","description","rol","roq","forecast_lt",
                        "avg_monthly_demand","stock_on_hand","policy_tier"]
        display_cols = [c for c in display_cols if c in flagged.columns]
        print("
Sample flagged SKUs (ROL or ROQ > 3x recent demand):")
        print(flagged[display_cols].head(20).to_string(index=False))
else:
    print("sanity_flag column not found in inventory_policy.")


## UIO-Based Plan Tab

In [ ]:
if len(uio_plan)>0:
    print(f"UIO-based plan: {len(uio_plan):,} parts")
    print(uio_plan.head(10).to_string())
    fig,ax = plt.subplots(figsize=(11,4))
    if "uio_demand_leadtime" in uio_plan.columns:
        uio_plan["uio_demand_leadtime"].clip(upper=uio_plan["uio_demand_leadtime"].quantile(0.95)).hist(
            bins=50,ax=ax,color=PALETTE[4],edgecolor="white",alpha=0.8)
        ax.set_title("UIO-Based Lead-Time Demand Distribution
"
                     "(replacement_freq × projected_UIO × 0.60 / 12 × 3)")
        ax.set_xlabel("Units (3-month)"); plt.tight_layout(); plt.show()
        top = uio_plan.nlargest(15,"uio_demand_leadtime")[["material_9","description","uio_demand_leadtime","compatible_models"]]
        print("
Top 15 by UIO-based demand:")
        print(top.to_string(index=False))


## Purchase Recommendation (Module 5)

In [ ]:
try:
    rec = load("m5_import_recommendation.parquet")
    print(f"Import recommendations: {len(rec):,} lines | cols: {rec.columns.tolist()}")
    print(rec.head(10).to_string())
    val_col = next((c for c in ["order_value_lkr","value_lkr","total_value"] if c in rec.columns),None)
    if val_col:
        total_val = rec[val_col].sum()
        print(f"
Total recommended order value: {fmt_lkr(total_val)}")
        fig,ax = plt.subplots(figsize=(11,4))
        rec[val_col].clip(upper=rec[val_col].quantile(0.95)).hist(bins=60,ax=ax,color=PALETTE[0],edgecolor="white",alpha=0.8)
        ax.set_title("Recommended Order Value per SKU (LKR, clipped 95th pct)")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e3:.0f}K"))
        plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("M5 import recommendation not yet computed. Run Module 5 from the Pipeline page.")

try:
    cr = load("m5_constrained_order.parquet")
    print(f"Constrained order: {len(cr):,} | cols: {cr.columns.tolist()}")
    print(cr.head(5).to_string())
except FileNotFoundError:
    pass
